<a href="https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper compares pages with rising impressions against pages with falling impressions. It reports that growing content tends to be longer, younger, and slightly better positioned in search. The paper reports that growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days). It also describes this as an observational comparison.

**Methodology question:** How were “rising” and “falling” labels defined from the impression data, and what exact reporting windows were used to create those labels? I would also want to understand whether the same content records or time periods are used across the comparison and validation evidence. Because this is an observational comparison, I would treat the reported relationships as directional rather than evidence that content length or age directly causes growth or decline.

### Finding 2 — The Content Performance Curve

The paper reports that content performance peaks at 61–90 days, declines after 270 days, and that the 365+ rebound is concentrated in older pages that were refreshed.

**Methodology question:** How was the performance curve calculated across the age ranges, and how was the “rebound” for refreshed older pages identified? I would also ask whether the analysis separates content age from refresh history and whether the validation design keeps different content groups or time periods separated. This would help determine whether the reported pattern is a directional portfolio observation or whether stronger evidence would be needed for a broader causal claim about content age or refreshing.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before/after validation

In Week 5, I evaluated the Random Forest using a stratified train/test split. For this audit, I re-evaluate the same model using a more honest validation design that keeps related observations separated by client.

The original evaluation is treated as the “before” result, while the grouped evaluation is the “after” result. The purpose is to check whether the model's measured decision-support performance remains similar when the validation design better reflects how the model may be used on unseen clients.


In [21]:
!git clone https://github.com/aliza1800/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [23]:
# Check the number of unique clients
print("Unique clients:", df["client_id"].nunique())

# Show how many records belong to each client
print("\nRecords per client:")
print(df["client_id"].value_counts().sort_index())

Unique clients: 32

Records per client:
client_id
client_02d20bbd7e      38
client_0b918943df      35
client_19581e27de    7008
client_1a6562590e       3
client_25fc0e7096     476
client_2c624232cd     649
client_349c41201b     763
client_3fdba35f04    2267
client_434c9b5ae5      87
client_4e07408562    2294
client_4ec9599fc2     556
client_4fc82b26ae      32
client_6208ef0f77    3681
client_624b60c58c     337
client_7f2253d7e2    1043
client_8527a891e2    1194
client_8722616204     578
client_8b940be7fb      28
client_9400f1b21c     158
client_98a3ab7c34     118
client_9f14025af0     325
client_a88a7902cb    1171
client_b4944c6ff0     254
client_bbb965ab0c     505
client_bdd2d3af3a      44
client_d029fa3a95     952
client_d4735e3a26    1106
client_d59eced1de      43
client_e29c9c180c     708
client_e629fa6598     720
client_f369cb89fc    1796
client_f74efabef1    1031
Name: count, dtype: int64


In [24]:
from sklearn.model_selection import GroupShuffleSplit

# Keep the same target used in Week 5
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Same Week-5 feature set
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features]
y = df["is_declining_label"]
groups = df["client_id"]

# Split by client: no client appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print("\nClient overlap:",
      len(set(df.iloc[train_idx]["client_id"]) &
          set(df.iloc[test_idx]["client_id"])))

Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap: 0


In [25]:
from sklearn.ensemble import RandomForestClassifier

# Train the Week-5 Random Forest on the grouped training data
rf_grouped = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_grouped.fit(X_train, y_train)

# Probability of the declining class
test_proba = rf_grouped.predict_proba(X_test)[:, 1]

print("Grouped Random Forest trained successfully.")

Grouped Random Forest trained successfully.


In [26]:
import numpy as np

def precision_at_k(y_true, scores, k):
    top_k = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k].mean()

# Honest grouped-split results
grouped_p20 = precision_at_k(y_test.reset_index(drop=True),
                              test_proba, 20)

grouped_p50 = precision_at_k(y_test.reset_index(drop=True),
                              test_proba, 50)

# Week-5 original results
week5_p20 = 0.900
week5_p50 = 0.920

print("Before (Week-5 stratified split):")
print(f"Precision@20: {week5_p20:.3f}")
print(f"Precision@50: {week5_p50:.3f}")

print("\nAfter (grouped-by-client split):")
print(f"Precision@20: {grouped_p20:.3f}")
print(f"Precision@50: {grouped_p50:.3f}")

Before (Week-5 stratified split):
Precision@20: 0.900
Precision@50: 0.920

After (grouped-by-client split):
Precision@20: 0.700
Precision@50: 0.740


### Before vs. after results

The Week-5 stratified split measured Precision@20 of 0.900 and Precision@50 of 0.920.

Under the grouped-by-client split, Precision@20 was 0.700 and Precision@50 was 0.740.

The grouped validation gives a more conservative measurement because no client appears in both the training and test sets. The measured precision decreased compared with the Week-5 stratified split. This suggests that the original results may not fully represent performance on unseen clients.

For this audit, I treat the grouped results as a more useful directional measure for decision-support on new clients. These results do not establish how the model would perform for every future client.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I audited the final Week-5 feature set for possible target leakage. The target is created from `trend_direction`, so the target column itself and `trend_pct` must not be used as model features.

The final feature set contains `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, and `word_count`. These features describe content characteristics or measured performance signals and are used as inputs rather than directly defining the target.

I also checked the dataset columns for fields that directly encode the target. `trend_direction` and `trend_pct` are excluded from the model feature list.

This audit does not prove that every feature is causally independent of the target. It confirms that the explicit target fields are not included in the final feature set.


In [27]:
# Final model features
print("Final model features:")
print(features)

# Columns directly related to the target
target_related = ["trend_direction", "trend_pct", "is_declining_label"]

print("\nTarget-related columns present in feature set:")
print([col for col in target_related if col in features])

# Check for exact target leakage
assert "trend_direction" not in features
assert "trend_pct" not in features
assert "is_declining_label" not in features

print("\nLeakage check passed: explicit target columns are not model features.")


Final model features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Target-related columns present in feature set:
[]

Leakage check passed: explicit target columns are not model features.


In [28]:
# Public-safe failure examples
# Identifiers are removed before displaying examples.

fp_examples = false_positives[
    features + ["actual", "predicted", "score"]
].head(5)

fn_examples = false_negatives[
    features + ["actual", "predicted", "score"]
].head(5)

print("Public-safe false positive examples:")
display(fp_examples)

print("\nPublic-safe false negative examples:")
display(fn_examples)

Public-safe false positive examples:


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,actual,predicted,score
22526,280,104,3445,39.0,0.09,1480.0,0,1,0.983333
4480,280,13,3171,31.8,0.03,2770.0,0,1,0.970000
12069,280,104,1677,33.1,0.18,1516.0,0,1,0.970000
27993,106,106,1266,4.6,0.00,2849.0,0,1,0.966667
2357,271,104,209,20.0,0.00,1450.0,0,1,0.963333



Public-safe false negative examples:


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,actual,predicted,score
27179,460,20,184,7.2,0.00,NaN,1,0,0.496667
9012,131,20,155,5.6,1.94,2755.0,1,0,0.496667
16180,276,20,16,3.5,0.00,4365.0,1,0,0.496667
13432,131,20,76,12.4,1.32,3080.0,1,0,0.496667
15300,347,7,1434,2.5,0.14,2553.0,1,0,0.496667


### Failure examples

The grouped test set contains 1,495 false positives and 1,190 false negatives at the classification threshold used for this audit.

The false-positive examples show cases where the model predicted decline (`predicted = 1`) but the observed label was non-declining (`actual = 0`). Several of these examples have relatively high model scores, showing that the model can be confident about cases that do not match the observed label.

The false-negative examples show cases where the model predicted non-decline (`predicted = 0`) while the observed label indicated decline (`actual = 1`). Their scores are close to the classification threshold, suggesting that some cases are difficult for the model to separate.

These examples show that the model's errors are not eliminated by the grouped validation design. They also reinforce that the model should be treated as decision-support rather than as a definitive classification system.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**
The Random Forest model can accurately identify content at risk of search-visibility decline.

**Rewritten claim:**
On the evaluated dataset, the Random Forest produced measured Precision@20 of 0.900 and Precision@50 of 0.920 under the Week-5 stratified split. Under a grouped-by-client validation design, the measured Precision@20 was 0.700 and Precision@50 was 0.740. These results indicate that the model can provide directional decision-support for prioritizing content for further review, but they do not establish reliable performance for every unseen client or future dataset.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.